In [2]:
!pip install streamlit pyngrok plotly --quiet


In [3]:
from pyngrok import ngrok

ngrok.set_auth_token("authtoken: 36NuAqsHghcjtz0HSuKAh6PTYm5_7GqEuckWxvLjgPfChL4pt")


In [9]:
!ls


app.py	sample_data


In [14]:
%%writefile app.py

import streamlit as st
import pandas as pd
import plotly.express as px
from sklearn.ensemble import IsolationForest

# ---------------- PAGE CONFIG ----------------
st.set_page_config(page_title="FitPulse Health Dashboard", layout="wide")

st.title("🏃 FitPulse Health Anomaly Detection Dashboard")

# ---------------- FILE UPLOAD ----------------
uploaded_file = st.file_uploader(
    "Upload Fitness Data (CSV or JSON)",
    type=["csv", "json"]
)

if uploaded_file is not None:

    # Load data
    if uploaded_file.name.endswith(".csv"):
        df = pd.read_csv(uploaded_file)
    else:
        df = pd.read_json(uploaded_file)

    st.subheader("📊 Uploaded Data Preview")
    st.dataframe(df.head())

    # ---------------- DATA PREPROCESSING ----------------
    df['date'] = pd.to_datetime(df['date'])

    # Convert wide → long format
    long_df = df.melt(
        id_vars=['date'],
        value_vars=['steps', 'heart_rate_avg', 'sleep_hours'],
        var_name='metric',
        value_name='value'
    )

    # Rename for consistency
    long_df.rename(columns={'date': 'timestamp'}, inplace=True)

    # ---------------- METRIC SELECTION ----------------
    metric = st.selectbox(
        "Select Health Metric",
        long_df['metric'].unique()
    )

    metric_df = long_df[long_df['metric'] == metric].copy()

    # ---------------- ANOMALY DETECTION ----------------
    model = IsolationForest(contamination=0.05, random_state=42)
    metric_df['anomaly'] = model.fit_predict(metric_df[['value']])

    # ---------------- VISUALIZATION ----------------
    fig = px.line(
        metric_df,
        x='timestamp',
        y='value',
        title=f"{metric.replace('_',' ').title()} Trend"
    )

    anomalies = metric_df[metric_df['anomaly'] == -1]

    fig.add_scatter(
        x=anomalies['timestamp'],
        y=anomalies['value'],
        mode='markers',
        marker=dict(color='red', size=8),
        name='Anomaly'
    )

    st.plotly_chart(fig, use_container_width=True)

    # ---------------- ANOMALY TABLE ----------------
    st.subheader("🚨 Detected Anomalies")
    st.dataframe(anomalies)

    # ---------------- REPORT DOWNLOAD ----------------
    csv = anomalies.to_csv(index=False)
    st.download_button(
        "⬇ Download Anomaly Report",
        csv,
        "anomaly_report.csv",
        "text/csv"
    )


Overwriting app.py


In [ ]:
from pyngrok import ngrok

ngrok.kill()

public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py &>/dev/null


NgrokTunnel: "https://snuffly-courageously-merissa.ngrok-free.dev" -> "http://localhost:8501"
